In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Mitygacja błędów za pomocą sieci tensorowych (TEM): Funkcja Qiskit firmy Algorithmiq
*Zobacz [dokumentację API](https://docs.quantum.ibm.com/api/functions/algorithmiq-tem)*

> **Note:** Funkcje Qiskit to eksperymentalna funkcjonalność dostępna wyłącznie dla użytkowników planów IBM Quantum&reg; Premium Plan, Flex Plan i On-Prem (przez IBM Quantum Platform API). Są w fazie podglądu przedpremierowego i mogą ulec zmianie.


<Accordion>
<AccordionItem title="Package versions">

Kod na tej stronie został opracowany przy użyciu poniższych wymagań.
Zalecamy korzystanie z tych lub nowszych wersji.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
## Przegląd
Metoda Mitygacji Błędów za pomocą Sieci Tensorowych (TEM) firmy Algorithmiq to hybrydowy
algorytm kwantowo-klasyczny zaprojektowany do przeprowadzania mitygacji szumów całkowicie
na etapie klasycznego post-processingu. Dzięki TEM możesz obliczać
wartości oczekiwane obserwowalnych, mitygując nieuniknione błędy wywoływane przez szumy,
które pojawiają się na sprzęcie kwantowym — z większą dokładnością i efektywnością kosztową,
co czyni tę metodę bardzo atrakcyjną zarówno dla badaczy kwantowych, jak i praktyków z przemysłu.

Metoda polega na skonstruowaniu sieci tensorowej reprezentującej odwrotność
globalnego kanału szumowego wpływającego na stan procesora kwantowego,
a następnie zastosowaniu tego odwzorowania do informacyjnie pełnych wyników pomiarów
uzyskanych z zaszumionego stanu w celu uzyskania nieobciążonych estymatorów obserwowalnych.

Zaletą TEM jest to, że wykorzystuje informacyjnie pełne pomiary, dając dostęp
do szerokiego zestawu mitygowanych wartości oczekiwanych obserwowalnych i osiągając
optymalny narzut próbkowania na sprzęcie kwantowym, zgodnie z opisem w Filippov et
al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), oraz Filippov
et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542). Narzut
pomiarowy odnosi się do liczby dodatkowych pomiarów wymaganych do
przeprowadzenia efektywnej mitygacji błędów — jest to kluczowy czynnik dla wykonalności
obliczeń kwantowych. Dlatego TEM ma potencjał, by umożliwić przewagę kwantową
w złożonych scenariuszach, takich jak zastosowania w dziedzinie chaosu kwantowego,
fizyki wielu ciał, dynamiki Hubbarda oraz symulacji chemii małych cząsteczek.

Główne cechy i korzyści TEM można podsumować następująco:

1.  **Optymalny narzut pomiarowy**: TEM jest optymalna względem
[granic teoretycznych](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
co oznacza, że żadna metoda nie może osiągnąć mniejszego narzutu pomiarowego. Innymi
słowy, TEM wymaga minimalnej liczby dodatkowych pomiarów do przeprowadzenia
mitygacji błędów. Przekłada się to bezpośrednio na minimalne zużycie czasu kwantowego.
2.  **Efektywność kosztowa**: Ponieważ TEM obsługuje mitygację szumów całkowicie w etapie
post-processingu, nie ma potrzeby dodawania dodatkowych obwodów do komputera kwantowego,
co nie tylko obniża koszty obliczeń, ale także zmniejsza ryzyko wprowadzenia dodatkowych
błędów wynikających z niedoskonałości urządzeń kwantowych.
3.  **Estymacja wielu obserwowalnych**: Dzięki informacyjnie pełnym pomiarom
TEM efektywnie estymuje wiele obserwowalnych na podstawie tych samych danych pomiarowych
z komputera kwantowego.
4.  **Mitygacja błędów odczytu**: Funkcja Qiskit TEM zawiera również
[zastrzeżoną metodę mitygacji błędów odczytu](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154),
zdolną do znacznego zmniejszenia błędów readout po krótkim przebiegu kalibracyjnym.
5.  **Dokładność**: TEM znacząco poprawia dokładność i niezawodność cyfrowych symulacji kwantowych, czyniąc algorytmy kwantowe bardziej precyzyjnymi i wiarygodnymi.
## Opis
Funkcja TEM pozwala uzyskać wartości oczekiwane z mitygacją błędów dla
wielu obserwowalnych w obwodzie kwantowym przy minimalnym narzucie próbkowania. Obwód
jest mierzony za pomocą informacyjnie pełnej miary z operatorem o wartości dodatniej (IC-POVM),
a zebrane wyniki pomiarów są przetwarzane na komputerze klasycznym. Ten pomiar służy do
wykonywania metod sieci tensorowych i zbudowania mapy odwrotności szumu. Funkcja stosuje
odwzorowanie, które w pełni odwraca cały zaszumiony obwód za pomocą sieci tensorowych
reprezentujących zaszumione warstwy.

![Schemat TEM](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Mitygowana błędem estymacja obserwowanej wartości O poprzez post-processing wyników pomiarów zaszumionego procesora kwantowego. U i N oznaczają idealną operację kwantową i powiązaną mapę szumu, która może być ogólnie nielokalna (i rozszerzona do szarych ramek). D oznacza tensor operatorów dualnych do efektów w pomiarze IC. Moduł mitygacji szumu M jest siecią tensorową efektywnie kontrahowaną od środka na zewnątrz. Pierwsza iteracja kontrakcji jest przedstawiona linią kropkowaną w kolorze fioletowym, druga linią przerywaną, a trzecia linią ciągłą.")

Po przesłaniu obwodów do funkcji są one transpilowane i
optymalizowane w celu zminimalizowania liczby warstw z bramkami dwu-qubitowymi (głośniejszymi
bramkami na urządzeniach kwantowych). Szum wpływający na warstwy jest uczony przy użyciu
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
z zastosowaniem rzadkiego modelu szumu Pauli-Lindblad, zgodnie z opisem w E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

Model szumu jest dokładnym opisem szumu na urządzeniu, zdolnym do
uchwycenia subtelnych cech, w tym przesłuchów między qubitami. Jednak szum na
urządzeniach może fluktuować i dryfować, a nauczony szum może nie być dokładny
w momencie przeprowadzania estymacji. Może to prowadzić do niedokładnych wyników.
## Pierwsze kroki
Uwierzytelnij się za pomocą [klucza API IBM Quantum Platform](http://quantum.cloud.ibm.com/) i wybierz funkcję TEM w następujący sposób. (Ten fragment kodu zakłada, że masz już [zapisane konto](/guides/functions#install-qiskit-functions-catalog-client) w środowisku lokalnym.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## Przykład
Poniższy fragment kodu pokazuje przykład, w którym TEM jest używana do obliczenia wartości oczekiwanych obserwowalnej dla prostego obwodu kwantowego.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device
# reported by IBM if not specified
backend_name = "ibm_marrakesh"

# Run the TEM function (uses around three minutes of QPU time)
job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Użyj interfejsów API Qiskit Serverless, aby sprawdzić status zadania funkcji Qiskit:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs
print(evs[0])

0.02165380888171687


Możesz pobrać wyniki w następujący sposób: